# Extracting Usable Motivations from Judicial Decision JSON Files

This script performs the following steps to build a corpus of motivations from judicial decisions:

## 1. Library Imports

The script imports several modules:
- Standard libraries: `re`, `os`, `json`, `pathlib`, etc.
- NLP and ML libraries: `torch`, `transformers`, `sklearn`, etc.
- Utilities: `tqdm` for progress tracking, `pandas` for data manipulation, etc.
- A mapping table (`MONTH_MAP`) converts French month names to numeric format (`"janvier" -> "01"`), useful for date handling.

## 2. Motivation Zone Extraction

Function `extract_motivation(data: dict)`:
- Takes a structured judicial decision in JSON format.
- Extracts the **motivation** zone using `"start"` and `"end"` indices.
- Ignores cases where:
  - Indices are invalid,
  - The `"motivations"` zone is absent or empty.

## 3. Traversing a Folder of Judicial Decision JSON Files

Function `process_all_json_with_motivation_only(json_dir)`:
- Recursively traverses all JSON files in a given folder.
- For each file:
  - Checks for the presence of key fields `"number"` and `"decision_date"`.
  - Uses `extract_motivation()` to extract the motivation.
  - Filters out unusable motivations (notably those containing:
  **"[NON-PUBLIC PROCEEDINGS - Decision motivation redacted]"**).
- Returns a dictionary of filtered decisions, keyed by `(number, decision_date)` with the motivation as value.

## 4. Corpus Loading

The folder `raw_decisions` contains all JSON decisions:
```python
json_dir = "artifacts/raw_decisions"
```

In [ ]:
import re
import os
import json
import numpy as np
import pandas as pd
import unicodedata
from typing import List, Dict, Tuple, Any, Optional, Union
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import CamembertForSequenceClassification, AutoTokenizer

# --- Extraction and chunking utilities ---

# Mapping table: French month names -> numeric ISO format
MONTH_MAP = {
    'janvier': '01', 'février': '02', 'mars': '03', 'avril': '04',
    'mai': '05', 'juin': '06', 'juillet': '07', 'août': '08',
    'septembre': '09', 'octobre': '10', 'novembre': '11', 'décembre': '12'
}

def extract_motivation(data: dict) -> str:
    """
    Extract only the 'motivation' zone from a decision JSON,
    using the start/end indices in the 'zones' section.
    """
    zones = data.get("zones", {})
    if not zones.get("motivations"):
        return ""
    mot = zones["motivations"][0]
    start, end = mot.get("start"), mot.get("end")
    texte = data.get("text", "")
    if start is None or end is None or end > len(texte):
        return ""
    return texte[start:end].strip()

def process_all_json_with_motivation_only(json_dir: Union[str, Path]) -> Dict[Tuple[str, str], Dict[str, Any]]:
    """
    Traverse all JSON files in a folder.
    Keep those with a 'number', 'decision_date', and a non-empty 'motivations' zone,
    excluding motivations containing the redaction notice.
    Returns a dict keyed by (number, decision_date) with motivation as value.
    """
    filtered = {}
    for root, _, files in os.walk(json_dir):
        for fname in files:
            if not fname.endswith('.json'):
                continue
            path = Path(root) / fname
            try:
                data = json.loads(path.read_text(encoding='utf8'))
            except Exception:
                continue  # skip unreadable or corrupted files

            num = data.get('number', '').strip()
            date = data.get('decision_date', '').strip()
            if not num or not date:
                continue

            mot = extract_motivation(data)
            if mot and "[DÉBATS NON PUBLICS – Motivation de la décision occultée]" not in mot:
                # filter out redacted motivations
                filtered[(num, date)] = {'motivation': mot}
    return filtered

# --- Load and run the pipeline ---

# directory containing all judicial decision JSON files
json_dir = "artifacts/raw_decisions"  # not shipped — regenerated by this step (see DATA.md)

# load all decisions with a usable motivation, without prior filtering
all_decisions_mixed = process_all_json_with_motivation_only(json_dir)

len(all_decisions_mixed)

**Purpose:**
Split text into **sentences** then into **chunks** of tokens (max 100 tokens) for NLP processing, while avoiding cuts in the middle of a sentence or abbreviation.

Main features
--------------

- **split_sentences**:
  - Splits text on `.`, `!`, `?` followed by a space.
  - Automatically rejoins splits caused by common abbreviations (e.g., "M.", "Dr.").
  - Result: clean list of sentences.

- **chunk_text_token_ids_grouped**:
  - For each sentence, encodes into tokens (via the provided tokenizer: CamemBERT, LegalTokenizer, etc.).
  - Groups sentences by 2 **if the total does not exceed 100 tokens** (otherwise splits sentence by sentence).
  - Also handles cases without punctuation or overly long sentences: direct split every 100 tokens.
  - Returns the list of chunks (text, not token_ids).

Parameters & usage
------------------

- **MAX_TOKENS**: max chunk size (default: 100 tokens).
- Accounts for abbreviations to avoid splitting at the wrong location.
- Compatible with all HuggingFace Transformers tokenizers.

In [ ]:
import re
from typing import List
from transformers import PreTrainedTokenizer
from transformers import RobertaTokenizerFast

# maximum number of tokens per chunk
MAX_TOKENS = 100

# JuriBERT tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained('JuriBERT/LegalTokenizer')  # external pretrained JuriBERT model, not shipped - see DATA.md


# abbreviations that should not be treated as sentence endings
_ABBREVIATIONS = {
    "M.", "Mme.", "Mlle.", "Dr.", "Pr.", "Me.", "St.", "Sr.",
    "Art.", "art.", "al.", "chap.", "n°", "R.", "L.", "r.", "l.",
    "cf.", "etc.", "ex.", "env.", "approx.", "min.", "max.",
    "sup.", "inf.", "p.", "pp.", "v.", "éd.", "trad.", "loc. cit.",
    "op. cit.", "ibid.", "e.g.", "i.e.", "av.", "apr.", "tps.",
    "qté.", "fig.", "litt.", "adj.", "subst.", "vb.", "tél.", "fax.",
    "C. civ.", "C. pén.", "C. trav.", "réf.", "C.", "com.", "Com.", "Civ."
}


# similarity thresholds (used in other parts of the pipeline)
SIM_THRESHOLD = 0.15  # minimum similarity to retain a pair
NEG_THRESHOLD = 0.05  # maximum for a negative chunk

# regex to naively split on . ! ? followed by a space
_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text: str) -> List[str]:
    """
    Split text into sentences, rejoining when an abbreviation causes a false split.
    Returns a corrected list of sentences.
    """
    parts = _SENTENCE_SPLIT.split(text)  # naive split
    sentences = []
    buffer = ""
    for part in parts:
        segment = part.strip()
        if not segment:
            continue  # skip empty segments
        # try to rejoin if split after an abbreviation
        candidate = buffer + " " + segment if buffer else segment
        last_word = candidate.split()[-1]
        # if last word is an abbreviation, rejoin
        if last_word in _ABBREVIATIONS or last_word.endswith(tuple(_ABBREVIATIONS)):
            buffer = candidate
        else:
            sentences.append(candidate)
            buffer = ""
    # append remainder if needed
    if buffer:
        sentences.append(buffer)
    return sentences

def chunk_text_token_ids_grouped(
    text: str,
    tokenizer: PreTrainedTokenizer,
    max_tokens: int = MAX_TOKENS
) -> List[str]:
    """
    Split text into chunks of <= max_tokens (BPE tokens), intelligently grouping
    sentences by two when possible. Also handles cases without punctuation.
    Returns a list of text chunks.
    """
    text = text.strip().replace("\n", " ").replace("  ", " ")  # basic cleanup

    # case where there is no punctuation to split on (raw text)
    if not any(p in text for p in ".!?"):
        token_ids = tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=512)
        # cut into max_tokens pieces
        return [
            tokenizer.decode(token_ids[i:i + max_tokens], skip_special_tokens=True)
            for i in range(0, len(token_ids), max_tokens)
        ]

    # 1. split into corrected sentences
    sentences = [s.strip() for s in split_sentences(text) if s.strip()]

    # 2. tokenize each sentence
    sent_token_ids = [
        tokenizer.encode(sent, add_special_tokens=False, truncation=True, max_length=512)
        for sent in sentences
    ]

    # 3. intelligently group (by 2 sentences when possible)
    chunks = []
    i = 0
    while i < len(sent_token_ids):
        ids_1 = sent_token_ids[i]
        ids_2 = sent_token_ids[i + 1] if i + 1 < len(sent_token_ids) else []
        # merge if sum fits in max_tokens
        if ids_2 and len(ids_1) + len(ids_2) <= max_tokens:
            merged = ids_1 + ids_2
            chunks.append(tokenizer.decode(merged, skip_special_tokens=True))
            i += 2
        elif len(ids_1) <= max_tokens:
            chunks.append(tokenizer.decode(ids_1, skip_special_tokens=True))
            i += 1
        else:
            # single sentence too long: cut to max size
            for j in range(0, len(ids_1), max_tokens):
                chunk = ids_1[j:j + max_tokens]
                chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
            i += 1
    return chunks

## Script: `export_chunks_only`

This script processes a corpus of judicial decisions by extracting the **motivation** from each decision, then **intelligently splitting it into chunks** (limited to a maximum number of tokens). It then exports the chunks in multiple formats for future use.

---

### Function: `export_chunks_only(decisions, tokenizer, output_basename, max_tokens=100)`

#### Parameters:
- `decisions`: dictionary `{(num, date): {"motivation": ...}}` obtained from a JSON corpus (via `process_all_json_with_motivation_only`)
- `tokenizer`: a HuggingFace-compatible tokenizer (e.g., JuriBERT) for token-based splitting
- `output_basename`: base path without extension (`str`) for exported files
- `max_tokens`: maximum number of tokens per chunk (default: `100`)

---

### Processing steps:

1. **Iterate over each decision**
   - Retrieve the motivation
   - Split into chunks of `max_tokens` via `chunk_text_token_ids_grouped`
   - Mask sensitive references in each chunk via `mask_sensitive_references`

2. **Create a `pandas` DataFrame** with columns:
   - `decision_num`
   - `decision_date`
   - `chunk_id`
   - `chunk_text`
   - `chunk_text_masked`

---

### Export results in 3 formats:

1. **CSV** - No size limitations - File: `output_basename.csv`
2. **Excel** - If < 1M rows: `output_basename.xlsx`; otherwise split into multiple files (`_part1.xlsx`, `_part2.xlsx`, ...)
3. **Parquet** - Compact and fast format - File: `output_basename.parquet`

In [ ]:
import pandas as pd
import os
from tqdm.auto import tqdm


# ----------------------- 1) Regex preparation --------------------------
CODES = [
    "procédures civiles d'exécution", "procédure civile", "procédure pénale",
    "organisation judiciaire", "juridictions financières", "justice administrative",
    "justice militaire", "justice pénale des mineurs",
    "action sociale et des familles", "famille et aide sociale",
    "pensions civiles et militaires de retraite",
    "pensions militaires d'invalidité et des victimes de guerre",
    "santé publique", "sécurité sociale", "travail", "commerce",
    "artisanat", "assurances", "consommation",
    "construction et de l'habitation", "monétaire et financier",
    "mutualité", "postes et communications électroniques",
    "propriété intellectuelle", "tourisme",
    "communes", "cinéma et de l'image animée", "défense",
    "domaine de l'État", "douanes", "éducation", "électoral",
    "étrangers et droit d'asile", "expropriation pour cause d'utilité publique",
    "propriété des personnes publiques", "collectivités territoriales",
    "général des impôts", "procédures fiscales", "patrimoine", "recherche",
    "route", "sécurité intérieure", "service national", "sport", "urbanisme",
    "voirie routière", "commande publique", "fonction publique",
    "transports", "aviation civile", "navigation intérieure",
    "pensions marins français", "ports maritimes", "emploi maritime",
    "environnement", "énergie", "minier", "forestier",
    "pêche maritime", "agricole et pastoral",
    "déontologie police", "déontologie municipales", "déontologie architectes",
    "marine marchande", "CESEDA", "entrée et du séjour des étrangers et du droit d'asile"
]
escaped_codes = sorted((re.escape(c) for c in CODES), key=len, reverse=True)
codes_full = rf"(?:code\s+(?:de|du|des|de\s+la|de\s+l'|d')?\s*(?:{'|'.join(escaped_codes)}))"
codes_abbr = r"(?:C\.?\s*(?:civ\.?|com\.?|consom\.?|trav\.?|assur\.?|proc\.\s*civ\.?))"
code_pattern = rf"(?:{codes_full}|{codes_abbr}|code\s+civil\b)"
code_full_regex = re.compile(code_pattern, flags=re.I)

REF_PATTERN = re.compile(rf"""
    \b
    (?:(?:des?|de\s+l')\s+)?
    art(?:icles?)?\.?\s*
    (?P<version>[LRD])?\.?\s*
    (?P<number>
        \d{{1,4}}(?:[-\u2013]\d+|\.\d+)*
        (?:\s*à\s*\d{{1,4}}(?:[-\u2013]\d+|\.\d+)*)?
        (?:\s*(?:,|\bet\b)\s*
            \d{{1,4}}(?:[-\u2013]\d+|\.\d+)*(?:\s*à\s*\d{{1,4}}(?:[-\u2013]\d+|\.\d+)*)?
        )*
        (?:\s+et\s+suivants)?
        (?:\s+ancien)?
    )
    (?:\s*(?:alinéa|§|I{{1,3}}|IV|V)\s*\d*)?
    (?:\s*(?:du|de|dans)\s*{code_pattern})?
""", re.I | re.VERBOSE)
article_full_regex = re.compile(REF_PATTERN.pattern, flags=re.I | re.VERBOSE)

loi_num_regex = re.compile(r"\bloi\s+(n°\s*)?\d{2,4}[-\u2013]?\d+(\s+du\s+\d{1,2}(?:er)?\s+\w+\s+\d{4})?", flags=re.I)
loi_date_regex = re.compile(r"(la|l['\u2019]?)\s+loi\s+du\s+\d{1,2}(?:er)?\s+\w+\s+\d{4}", flags=re.I)

money_regex = re.compile(r'(?<!\d)(\d+(?:[ \.,]\d+)*)(?=\s*(?:€|euros?))\s*(?:€|euros?)', flags=re.I)

# ----------------------- 2) Utility functions --------------------------
def mask_sensitive_references(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = article_full_regex.sub("[ARTICLE]", text)
    text = code_full_regex.sub("[CODE]", text)
    text = loi_num_regex.sub("[LOI]", text)
    text = loi_date_regex.sub("[LOI]", text)
    text = money_regex.sub("[MONTANT]", text)
    return text

def export_chunks_only(decisions: dict, tokenizer, output_basename: str, max_tokens: int = 100) -> pd.DataFrame:
    """
    Split motivations into chunks and export results:
    - as CSV (no limit)
    - as multiple Excel files (if > 1M rows)
    - as Parquet (compact and fast format)
    """
    results = []

    for (num, date), data in tqdm(decisions.items(), desc="Chunking"):
        motivation = data.get("motivation", "")
        if not motivation:
            continue

        chunks = chunk_text_token_ids_grouped(motivation, tokenizer, max_tokens=max_tokens)

        for i, chunk in enumerate(chunks):
            results.append({
                "decision_num": num,
                "decision_date": date,
                "chunk_id": i,
                "chunk_text": chunk,
                "chunk_text_masked": mask_sensitive_references(chunk),
            })

    # convert to DataFrame
    df = pd.DataFrame(results)

    # add decision_id column (num__date)
    df["decision_id"] = df["decision_num"].astype(str) + "__" + df["decision_date"].astype(str)

    # create output directory if needed
    os.makedirs(os.path.dirname(output_basename), exist_ok=True)

    # ---------- 1. CSV export (complete) ----------
    csv_path = output_basename + ".csv"
    df.to_csv(csv_path, index=False)
    print(f"CSV exported: {csv_path}")

    # ---------- 2. Excel export (split if necessary) ----------
    MAX_ROWS = 1_000_000
    if len(df) <= MAX_ROWS:
        xlsx_path = output_basename + ".xlsx"
        df.to_excel(xlsx_path, index=False)
        print(f"Excel exported: {xlsx_path}")
    else:
        for i in range(0, len(df), MAX_ROWS):
            part = df.iloc[i:i + MAX_ROWS]
            part_path = output_basename + f"_part{i // MAX_ROWS + 1}.xlsx"
            part.to_excel(part_path, index=False)
            print(f"Excel part exported: {part_path}")

    # ---------- 3. Parquet export ----------
    parquet_path = output_basename + ".parquet"
    df.to_parquet(parquet_path, index=False)
    print(f"Parquet exported: {parquet_path}")

    return df

In [ ]:
export_chunks_only(
 decisions=all_decisions_mixed,
 tokenizer=tokenizer,
 output_basename="artifacts/chunks/all_chunks"  # not shipped — regenerated by this step (see DATA.md)
)


In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# load data
df = pd.read_parquet("artifacts/chunks/all_chunks.parquet")  # not shipped — regenerated by this step (see DATA.md)

# (re)create the decision_id column if it does not exist
if "decision_id" not in df.columns:
    df["decision_id"] = df["decision_num"].astype(str) + "__" + df["decision_date"].astype(str)

# extract unique decision_id list
decision_ids = df["decision_id"].unique()

# fix seed for reproducibility
SEED = 42
train_ids, tmp_ids = train_test_split(decision_ids, test_size=0.30, random_state=SEED)
val_ids, test_ids = train_test_split(tmp_ids, test_size=0.50, random_state=SEED)

# split dataframe rows by membership
df_train = df[df["decision_id"].isin(train_ids)].reset_index(drop=True)
df_val = df[df["decision_id"].isin(val_ids)].reset_index(drop=True)
df_test = df[df["decision_id"].isin(test_ids)].reset_index(drop=True)

# create output directory if needed
outdir = "artifacts/chunks"
os.makedirs(outdir, exist_ok=True)

# save subsets
df_train.to_parquet(f"{outdir}/chunks_train.parquet", index=False)
df_val.to_parquet(f"{outdir}/chunks_val.parquet", index=False)
df_test.to_parquet(f"{outdir}/chunks_test.parquet", index=False)

print("Data split by decision_id and saved:")
print(f"  - Train: {len(df_train)} chunks")
print(f"  - Val:   {len(df_val)} chunks")
print(f"  - Test:  {len(df_test)} chunks")

# Parallel Batch TF-IDF Chunk-to-Article Matching (Training Set)

---

## 1) Data Loading
- **Civil Code**: loaded from a JSON file.
- **Decision chunks**: loaded from `chunks_train.parquet`.

---

## 2) TF-IDF Vectorization
- Uses `TfidfVectorizer` with: `min_df=2`, `ngram_range=(1,2)`, `sublinear_tf=True`
- Vocabulary learned on all texts (articles + chunks).
- `A` = TF-IDF matrix of articles (comparison base)
- Chunks are vectorized dynamically per batch.

---

## 3) Batch Processing
- The chunk DataFrame is split into **batches of 10,000 rows**.

### Function `process_batch(batch_df, batch_idx)`:
- Transforms each chunk into a TF-IDF vector
- Computes **cosine similarities** between each chunk and all articles
- If similarity >= 0.15 **and** article number is mentioned in the chunk -> `label = 1` (positive)
- A **negative article** (not mentioned, score < 0.05) is randomly selected for each positive.

---

## 4) Merge & Export
- All results are merged into a final DataFrame.
- Saved to: `tfidf_chunks_train.parquet`

In [ ]:
import os
import json
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

# load the Civil Code
print("Loading Civil Code...")
with open("DATA/inputs/civil_code_articles.json", encoding="utf-8") as f:
    articles = json.load(f)

# load parquet file
input_path = "artifacts/chunks/chunks_train.parquet"  # not shipped — regenerated by this step (see DATA.md)
print(f"Loading chunks from: {input_path}")
df_chunks = pd.read_parquet(input_path)
print(f"{len(df_chunks):,} chunks loaded.")

# parameters
BATCH_SIZE = 10000
N_JOBS = -1

# TF-IDF vectorization
print("Learning TF-IDF vectorizer on articles + chunks...")
article_ids = list(articles.keys())
article_texts = list(articles.values())
vect = TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)
vect.fit(article_texts + df_chunks["chunk_text"].tolist())
A = vect.transform(article_texts)
print("Article vectorization complete.")

# --- batch processing function
def process_batch(batch_df, batch_idx):
    print(f"  Processing batch {batch_idx} ({len(batch_df)} chunks)...")
    from random import choice
    chunk_texts = batch_df["chunk_text"].tolist()
    C = vect.transform(chunk_texts)
    sims = cosine_similarity(C, A)

    results = []
    for local_i, (idx, row) in enumerate(batch_df.iterrows()):
        chunk_txt = row["chunk_text"]
        for j, article_num in enumerate(article_ids):
            score = sims[local_i, j]
            if score >= 0.15 and article_num.lower() in chunk_txt.lower():
                results.append({
                    "decision_id": row["decision_id"],
                    "chunk_id": row["chunk_id"],
                    "chunk_text": chunk_txt,
                    "chunk_text_masked": row["chunk_text_masked"],
                    "article": article_num,
                    "article_text": articles[article_num],
                    "similarity": float(score),
                    "label": 1
                })
                neg_candidates = [
                    jj for jj in range(len(article_ids))
                    if sims[local_i, jj] < 0.05 and article_ids[jj].lower() not in chunk_txt.lower()
                ]
                if neg_candidates:
                    jj = choice(neg_candidates)
                    results.append({
                        "decision_id": row["decision_id"],
                        "chunk_id": row["chunk_id"],
                        "chunk_text": chunk_txt,
                        "chunk_text_masked": row["chunk_text_masked"],
                        "article": article_ids[jj],
                        "article_text": articles[article_ids[jj]],
                        "similarity": float(sims[local_i, jj]),
                        "label": 0
                    })
    print(f"  Batch {batch_idx} done: {len(results)} examples generated.")
    return results

# split into batches
batches = [df_chunks.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_chunks), BATCH_SIZE)]
print(f"{len(batches)} batches to process (batch size = {BATCH_SIZE})")

# parallel processing
all_results = Parallel(n_jobs=N_JOBS)(
    delayed(process_batch)(batch, idx) for idx, batch in enumerate(tqdm(batches, desc="Batch processing"))
)

# merge results
print("Merging results...")
flat_results = [row for batch_result in all_results for row in batch_result]
df_final = pd.DataFrame(flat_results)
print(f"{len(df_final):,} total examples.")

# save
output_dir = "artifacts/tfidf"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "tfidf_chunks_train.parquet")
df_final.to_parquet(output_file)
print(f"Result saved: {output_file}")

# Filtering Positive Examples (label = 1)

---

## 1) Load File
- Source: `tfidf_chunks_train.parquet` (contains labeled positive and negative examples)

## 2) Filter
- Keep only rows where `label == 1` (positive examples: chunks matched to an article)

## 3) Save
- Positive examples saved to: `tfidf_POS_chunks_train.parquet`

In [ ]:
import pandas as pd
import os

# source file
input_file = "artifacts/tfidf/tfidf_chunks_train.parquet"  # not shipped — regenerated by this step (see DATA.md)
df = pd.read_parquet(input_file)

# filter: positive examples only
df_pos = df[df["label"] == 1]

# output path
output_file_pos = os.path.join(
    "artifacts/tfidf",
    "tfidf_POS_chunks_train.parquet"
)

# save
df_pos.to_parquet(output_file_pos)

print(f"{len(df_pos):,} positive examples saved to: {output_file_pos}")

# Building the Validation Set: TF-IDF Chunk-to-Article Matching

---

## Data Loading
- **Civil Code**: articles loaded from a JSON file.
- **Validation chunks**: loaded from `chunks_val.parquet`.

---

## TF-IDF Vectorization
- Learns a TF-IDF vectorizer on all texts (articles + chunks).
- Articles are vectorized once.

---

## Batch Processing
- Chunks split into batches of 10,000.
- For each chunk:
  - Computes cosine similarity with each article.
  - If similarity >= 0.15 **and** article is mentioned -> **positive** example.
  - Adds a **negative** example (score < 0.05, article not mentioned).

---

## Merge & Export
- All results merged into a final DataFrame.
- Saved to: `tfidf_chunks_VALIDATION.parquet`

**Purpose**: Automatically generate the validation dataset for a supervised chunk-to-article matching task.

In [ ]:
import os
import json
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

# load the Civil Code
print("Loading Civil Code...")
with open("DATA/inputs/civil_code_articles.json", encoding="utf-8") as f:
    articles = json.load(f)

# load parquet file
input_path = "artifacts/chunks/chunks_val.parquet"  # not shipped — regenerated by this step (see DATA.md)
print(f"Loading chunks from: {input_path}")
df_chunks = pd.read_parquet(input_path)
print(f"{len(df_chunks):,} chunks loaded.")

# parameters
BATCH_SIZE = 10000
N_JOBS = -1

# TF-IDF vectorization
print("Learning TF-IDF vectorizer on articles + chunks...")
article_ids = list(articles.keys())
article_texts = list(articles.values())
vect = TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)
vect.fit(article_texts + df_chunks["chunk_text"].tolist())
A = vect.transform(article_texts)
print("Article vectorization complete.")

# --- batch processing function
def process_batch(batch_df, batch_idx):
    print(f"  Processing batch {batch_idx} ({len(batch_df)} chunks)...")
    from random import choice
    chunk_texts = batch_df["chunk_text"].tolist()
    C = vect.transform(chunk_texts)
    sims = cosine_similarity(C, A)

    results = []
    for local_i, (idx, row) in enumerate(batch_df.iterrows()):
        chunk_txt = row["chunk_text"]
        for j, article_num in enumerate(article_ids):
            score = sims[local_i, j]
            if score >= 0.15 and article_num.lower() in chunk_txt.lower():
                results.append({
                    "decision_id": row["decision_id"],
                    "chunk_id": row["chunk_id"],
                    "chunk_text": chunk_txt,
                    "chunk_text_masked": row["chunk_text_masked"],
                    "article": article_num,
                    "article_text": articles[article_num],
                    "similarity": float(score),
                    "label": 1
                })
                neg_candidates = [
                    jj for jj in range(len(article_ids))
                    if sims[local_i, jj] < 0.05 and article_ids[jj].lower() not in chunk_txt.lower()
                ]
                if neg_candidates:
                    jj = choice(neg_candidates)
                    results.append({
                        "decision_id": row["decision_id"],
                        "chunk_id": row["chunk_id"],
                        "chunk_text": chunk_txt,
                        "chunk_text_masked": row["chunk_text_masked"],
                        "article": article_ids[jj],
                        "article_text": articles[article_ids[jj]],
                        "similarity": float(sims[local_i, jj]),
                        "label": 0
                    })
    print(f"  Batch {batch_idx} done: {len(results)} examples generated.")
    return results

# split into batches
batches = [df_chunks.iloc[i:i+BATCH_SIZE] for i in range(0, len(df_chunks), BATCH_SIZE)]
print(f"{len(batches)} batches to process (batch size = {BATCH_SIZE})")

# parallel processing
all_results = Parallel(n_jobs=N_JOBS)(
    delayed(process_batch)(batch, idx) for idx, batch in enumerate(tqdm(batches, desc="Batch processing"))
)

# merge results
print("Merging results...")
flat_results = [row for batch_result in all_results for row in batch_result]
df_final = pd.DataFrame(flat_results)
print(f"{len(df_final):,} total examples.")

# save
output_dir = "artifacts/tfidf"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "tfidf_chunks_VALIDATION.parquet")
df_final.to_parquet(output_file)
print(f"Result saved: {output_file}")

# Filtering Positive Explicit Examples for Validation (label = 1)

---

## 1) Load File
- Source: `tfidf_chunks_VALIDATION.parquet` (contains labeled positive and negative examples)

## 2) Filter
- Keep only rows where `label == 1` (positive examples: chunks matched to an article)

## 3) Save
- Positive examples saved to: `tfidf_chunks_VALIDATION_POS_ONLY_explicit.parquet`

In [ ]:
import pandas as pd
import os

# source file
input_file = "artifacts/tfidf/tfidf_chunks_VALIDATION.parquet"  # not shipped — regenerated by this step (see DATA.md)
df = pd.read_parquet(input_file)

# filter: positive examples only
df_pos = df[df["label"] == 1]

# output path
output_file_pos = os.path.join(
    "artifacts/tfidf",
    "tfidf_chunks_VALIDATION_POS_ONLY_explicit.parquet"
)

# save
df_pos.to_parquet(output_file_pos)

print(f"{len(df_pos):,} positive examples saved to: {output_file_pos}")